# 03 · Does a Random Forest beat the burn rule?

**Project:** Paddy Burnt-Area Intelligence (Satellite Mission Intelligence for India)
**Question for this notebook:** does a machine-learning classifier, trained on the validation labels, map burnt fields better than Notebook 02's three-threshold rule?

**Why:** Notebook 02 calls a harvested field burnt when, in one 5-day composite, NBR ≤ −0.05, MIRBI ≥ 1.65 and NIR ≤ 0.13. The thresholds were set by eye on the pilot box and then checked against the labels. A classifier can weigh many more signals at once, such as how green the crop got, how dark the field turned and whether it greened again. If it does better, it should replace the rule; if not, the rule stays and the reasons are worth knowing. Checkpoint 3 also left one idea untested, an NDVI ceiling on the burn test; the forest sees NDVI at the darkest view and after it, so it can learn such a ceiling if one helps.

**How it works**
1. For every harvested pixel, the 18 composites from Notebook 02 are reduced to 23 numbers that describe its season around harvest: the green peak, the date of harvest, the darkest view after harvest and its bands, how far NBR dropped, and whether the field greened again (section 2).
2. A Random Forest (Breiman, 2001) learns burnt or not from the labelled points (`yes` and `no`; `unsure` points are left out). It is trained on half A and tested on half B, the same split used to check the rule, and also cross-validated on all labelled points, repeated 10 times.
3. Both methods are scored the same way as in Notebook 02, including the area-weighted accuracy of Stehman (2014), and an exact McNemar test (McNemar, 1947) checks whether their difference could be chance.
4. A learning curve shows whether more labels would help, and permutation importance shows what the forest relies on.
5. The forest's calls on the `unsure` points are set beside the rule's.
6. Optionally, the forest maps the whole district, and the map is compared with the rule's, pixel by pixel.

**Before you run it:** run Notebook 02 for the whole district (its composites must be in `data/raw/sentinel2/sangrur/district/`) and finish the validation labels. Nothing is downloaded and no key is needed.

**Run time:** about 5 minutes on a computer with 2 processor cores, less with more: about two-thirds for sections 5 to 7 (cross-validation, learning curve, importance) and a third for the district map.

**What it produces**

| Output | Where |
|---|---|
| Labels, both methods' calls and the features at the 240 points | `data/processed/ml_point_features_sangrur_2025.csv` |
| Accuracy of the forest and the rule, on half B and in cross-validation | `data/processed/ml_accuracy_sangrur_2025.csv` |
| Learning curve | `data/processed/ml_learning_curve_sangrur_2025.csv` |
| Feature importance | `data/processed/ml_feature_importance_sangrur_2025.csv` |
| Forest and rule maps compared: areas, edges, small patches | `data/processed/ml_map_comparison_sangrur_2025.csv` |
| Agreement map (GeoTIFF, 20 m) | `data/processed/ml_burn_agreement_sangrur_2025.tif` |
| Figures | `maps/notebook03/` |

In [ ]:
# Colab: add libraries its default image may lack. On your own computer, install requirements.txt into the
# project's environment first; there this cell only checks that the install worked.
import importlib.util, subprocess, sys

try:
    import google.colab  # noqa: F401  (this module exists only on Colab)
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

NEEDED = ('rasterio', 'sklearn', 'scipy', 'matplotlib', 'pandas')
if ON_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio'], check=True)
else:
    missing = [name for name in NEEDED if importlib.util.find_spec(name) is None]
    if missing:
        raise ModuleNotFoundError('Missing: ' + ', '.join(missing) + ". Install requirements.txt into the "
                                  "project's environment and re-run.")
    print('Libraries found.')

## 1. Settings

In [ ]:
import os, re, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Rectangle
import rasterio
from rasterio.windows import Window
from scipy import ndimage
from scipy.stats import binomtest
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from IPython.display import display

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/India-Space-Mission-Intelligence')
elif os.environ.get('PROJECT_DIR'):
    PROJECT = Path(os.environ['PROJECT_DIR']).resolve()
else:
    # On your computer: walk up from the working folder to the project root (the folder with requirements.txt).
    here = Path.cwd().resolve()
    PROJECT = next((p for p in (here, *here.parents)
                    if (p / 'requirements.txt').exists() and (p / 'notebooks').is_dir()), here.parent)

PROCESSED = PROJECT / 'data' / 'processed'
FIGS = PROJECT / 'maps' / 'notebook03'
FIGS.mkdir(parents=True, exist_ok=True)
print('Project folder:', PROJECT, '| scikit-learn', sklearn.__version__)

# --- Inputs from Notebook 02 ----------------------------------------------------------------------------
SUFFIX = 'sangrur_2025'
COMPOSITES = PROJECT / 'data' / 'raw' / 'sentinel2' / 'sangrur' / 'district'   # Notebook 02's 5-day composites
BANDS = ['B04', 'B08', 'B11', 'B12']    # red, NIR, SWIR 1, SWIR 2, as Notebook 02 downloads them

# --- The forest ------------------------------------------------------------------------------------------
N_TREES = 500
MIN_LEAF = 2                            # each leaf holds at least 2 training points: guards against memorising
SEED = 0
FOLDS, REPEATS = 5, 10                  # cross-validation: 5 folds, repeated 10 times with different splits
LEARNING_SIZES = (20, 40, 60, 80, 100)  # training-set sizes for the learning curve (plus the largest possible)
LEARNING_REPEATS = 10

# --- The district map ------------------------------------------------------------------------------------
MAKE_MAP = True                         # False skips section 8 (about half the run time)
CHUNK_ROWS = 512                        # rows of the district read at a time; lower it if memory runs short

## 2. Helper functions

**Season features.** Each pixel's composites are read in time order, as in Notebook 02. A pixel counts as harvested in the first window in which its NDVI falls below the harvest threshold after having reached the green threshold, with the thresholds Notebook 02 recorded. From that window on, the helper keeps running minima and maxima, and the bands of the darkest view (lowest NIR). Pixels never harvested get no features: both methods call them not burnt.

In [ ]:
FEATURES = ['peak_ndvi', 'harvest_day', 'pre_harvest_nbr', 'views_after_harvest', 'min_nbr', 'max_mirbi', 'min_nir',
            'min_red', 'min_swir2', 'nbr_drop', 'dark_red', 'dark_nir', 'dark_swir1', 'dark_swir2', 'dark_ndvi',
            'dark_nbr', 'dark_mirbi', 'dark_day', 'days_harvest_to_dark', 'ndvi_after_dark', 'nir_rise_after_dark',
            'max_ndvi_after_harvest', 'last_ndvi']


def dated_tifs(folder):
    files = {}
    for path in sorted(Path(folder).glob('*.tif')):
        found = re.search(r'(\d{4}-\d{2}-\d{2})', path.name)
        if found:
            files[found.group(1)] = path
    return files


def to_reflectance(raw, src):
    # As Notebook 02 reads the composites: the file's nodata value means no data, and so does 0 in a 16-bit file
    # (in a float file, 0 in every band). Then Notebook 02's offset and scale. raw: (bands, pixels).
    raw = raw.astype('float32')
    if src.nodata is not None and not np.isnan(src.nodata):
        raw[raw == src.nodata] = np.nan
    if np.issubdtype(np.dtype(src.dtypes[0]), np.integer):
        raw[raw == 0] = np.nan
    else:
        raw[:, np.all(raw == 0, axis=0)] = np.nan
    return (raw - DN_OFFSET) / SCALE


def band_order(src):
    names = [d or '' for d in src.descriptions]
    return [names.index(b) for b in BANDS] if all(b in names for b in BANDS) else list(range(len(BANDS)))


def season_features(windows, read):
    # windows: dates of the composites, in order; read(w) -> reflectance (4, n): red, NIR, SWIR 1, SWIR 2.
    # Returns the features (n, len(FEATURES)), NaN for pixels never harvested, and the harvested mask.
    days = np.array([pd.Timestamp(d).dayofyear for d in windows], 'float32')
    first = read(0)
    n = first.shape[1]

    def empty(value=np.nan):
        return np.full(n, value, 'float32')

    peak, harvest_w = empty(-np.inf), np.full(n, -1, 'int16')
    last_nbr, pre_nbr, views = empty(), empty(), empty(0.0)
    low = {k: empty(np.inf) for k in ('nbr', 'nir', 'red', 'sw2')}
    max_mirbi, max_ndvi, last_ndvi = empty(-np.inf), empty(-np.inf), empty()
    dark = {k: empty() for k in ('red', 'nir', 'sw1', 'sw2', 'ndvi', 'nbr', 'mirbi', 'day')}
    after_ndvi, after_nir, waiting = empty(), empty(), np.zeros(n, bool)
    for w in range(len(windows)):
        red, nir, sw1, sw2 = first if w == 0 else read(w)
        with np.errstate(divide='ignore', invalid='ignore'):
            ndvi = (nir - red) / (nir + red)
            nbr = (nir - sw2) / (nir + sw2)
        mirbi = 10 * sw2 - 9.8 * sw1 + 2
        valid = np.isfinite(ndvi) & np.isfinite(nbr) & np.isfinite(mirbi)
        newly = valid & (harvest_w < 0) & (peak >= RULE['green_ndvi']) & (ndvi < RULE['harvest_ndvi'])
        pre_nbr[newly] = last_nbr[newly]
        harvest_w[newly] = w
        after = valid & (harvest_w >= 0)
        got = after & waiting & ~newly                # first view after the darkest one so far
        after_ndvi[got], after_nir[got], waiting[got] = ndvi[got], nir[got], False
        views[after] += 1
        for key, value in (('nbr', nbr), ('nir', nir), ('red', red), ('sw2', sw2)):
            low[key][after] = np.minimum(low[key][after], value[after])
        max_mirbi[after] = np.maximum(max_mirbi[after], mirbi[after])
        max_ndvi[after] = np.maximum(max_ndvi[after], ndvi[after])
        darker = after & ~(nir >= dark['nir'])        # also true where there is no darkest view yet (NaN)
        for key, value in (('red', red), ('nir', nir), ('sw1', sw1), ('sw2', sw2), ('ndvi', ndvi), ('nbr', nbr),
                           ('mirbi', mirbi)):
            dark[key][darker] = value[darker]
        dark['day'][darker] = days[w]
        after_ndvi[darker], after_nir[darker], waiting[darker] = np.nan, np.nan, True
        last_ndvi[valid] = ndvi[valid]
        before = valid & (harvest_w < 0)
        last_nbr[before] = nbr[before]
        peak[valid] = np.maximum(peak[valid], ndvi[valid])
    harvested = harvest_w >= 0
    harvest_day = np.where(harvested, days[np.clip(harvest_w, 0, None)], np.nan)
    columns = {
        'peak_ndvi': peak, 'harvest_day': harvest_day, 'pre_harvest_nbr': pre_nbr, 'views_after_harvest': views,
        'min_nbr': low['nbr'], 'max_mirbi': max_mirbi, 'min_nir': low['nir'], 'min_red': low['red'],
        'min_swir2': low['sw2'], 'nbr_drop': pre_nbr - low['nbr'], 'dark_red': dark['red'], 'dark_nir': dark['nir'],
        'dark_swir1': dark['sw1'], 'dark_swir2': dark['sw2'], 'dark_ndvi': dark['ndvi'], 'dark_nbr': dark['nbr'],
        'dark_mirbi': dark['mirbi'], 'dark_day': dark['day'], 'days_harvest_to_dark': dark['day'] - harvest_day,
        'ndvi_after_dark': np.where(np.isfinite(after_ndvi), after_ndvi, dark['ndvi']),     # no later view: no change
        'nir_rise_after_dark': np.where(np.isfinite(after_nir), after_nir - dark['nir'], 0.0),
        'max_ndvi_after_harvest': max_ndvi, 'last_ndvi': last_ndvi}
    X = np.vstack([columns[k] for k in FEATURES]).T.astype('float32')
    X[~np.isfinite(X)] = np.nan
    X[~harvested] = np.nan
    return X, harvested


def ready(X):
    # The forest needs numbers everywhere: the rare gaps in a harvested pixel's features become 0.
    return np.nan_to_num(X, nan=0.0)


def forest(seed=SEED, trees=N_TREES):
    return RandomForestClassifier(n_estimators=trees, min_samples_leaf=MIN_LEAF, max_features='sqrt',
                                  class_weight='balanced', random_state=seed, n_jobs=-1)


def scores(pred, sel, truth):
    tp, fp, fn = (pred & truth & sel).sum(), (pred & ~truth & sel).sum(), (~pred & truth & sel).sum()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return precision, recall, (2 * precision * recall / (precision + recall) if precision + recall else 0.0)


def weighted_accuracy(pred, sel, truth, strata, weights):
    # Area-weighted user's, producer's and overall accuracy for 'burnt' (Stehman 2014), as in Notebook 02.
    cm, var_oa = np.zeros((2, 2)), 0.0
    for stratum, w_h in weights.items():
        in_h = (strata == stratum) & sel
        n_h = in_h.sum()
        if n_h == 0:
            continue
        for i, p in enumerate((True, False)):
            for j, tv in enumerate((True, False)):
                cm[i, j] += w_h * ((pred == p) & (truth == tv) & in_h).sum() / n_h
        correct = ((pred == truth) & in_h).sum() / n_h
        var_oa += w_h ** 2 * correct * (1 - correct) / max(n_h - 1, 1)
    ua = cm[0, 0] / cm[0].sum() if cm[0].sum() else np.nan
    pa = cm[0, 0] / cm[:, 0].sum() if cm[:, 0].sum() else np.nan
    return ua, pa, cm[0, 0] + cm[1, 1], np.sqrt(var_oa)


def score(pred, sel):
    p, r, f1 = scores(pred, sel, truth)
    ua, pa, oa, oa_se = weighted_accuracy(pred, sel, truth, strata, weights)
    return {'points': int(sel.sum()), 'precision': p, 'recall': r, 'f1': f1, 'users_accuracy_burnt': ua,
            'producers_accuracy_burnt': pa, 'overall_accuracy': oa, 'overall_accuracy_se': oa_se}


def mcnemar(a, b, sel):
    # Exact McNemar test: of the points exactly one method gets right, is the split far from 50:50?
    only_a = int(((a == truth) & (b != truth) & sel).sum())
    only_b = int(((b == truth) & (a != truth) & sel).sum())
    p = binomtest(only_b, only_a + only_b).pvalue if only_a + only_b else 1.0
    return only_a, only_b, p

## 3. The labelled points and their features

The validation points, their labels and Notebook 02's map come from Notebook 02's outputs. Each point is the centre pixel of a 3 × 3 block of one mapped class. The rule's call at each point is read from Notebook 02's class map: *map as reported* counts classes 3 and 4 as burnt; *every burn* also counts the unconfirmed late burns (class 5).

In [ ]:
summary = pd.read_csv(PROCESSED / f's2_burn_summary_{SUFFIX}.csv').set_index('metric')['value']
RULE = json.loads(summary['rule'])
found = re.search(r'offset\s+(-?[\d.]+),\s*scale\s+([\d.]+)', str(summary.get('scaling', '')))
if not found:
    raise RuntimeError("Notebook 02's summary has no 'scaling' row: run Notebook 02 for the whole district first.")
DN_OFFSET, SCALE = float(found.group(1)), float(found.group(2))
files = dated_tifs(COMPOSITES)
if len(files) < 2:
    raise RuntimeError(f'No composites in {COMPOSITES}: run Notebook 02 for the whole district first.')
WINDOWS = sorted(files)
print(f'{len(WINDOWS)} composites, {WINDOWS[0]} to {WINDOWS[-1]}; rule {RULE}')

points = pd.read_csv(PROCESSED / f'validation_points_{SUFFIX}.csv')
labels = pd.read_csv(PROCESSED / f'validation_labels_{SUFFIX}.csv', dtype=str).fillna('')
pts = points.merge(labels[['point_id', 'burnt', 'notes']], on='point_id', how='left').fillna({'burnt': ''})
answer = pts['burnt'].str.strip().str.lower().to_numpy()
sure = np.isin(answer, ['yes', 'no'])
truth = answer == 'yes'
strata = pts['stratum'].to_numpy()
weights = pts.groupby('stratum')['stratum_pixels'].first()
weights = weights / weights.sum()
half_a = sure & (pts['half'] == 'A').to_numpy()
half_b = sure & (pts['half'] == 'B').to_numpy()
if sure.sum() < 40 or truth[sure].sum() < 10:
    raise RuntimeError(f'Too few labelled points ({sure.sum()}, {truth[sure].sum()} burnt): finish the labels first.')

with rasterio.open(PROCESSED / f's2_burn_classes_{SUFFIX}.tif') as src:
    CLASS_MAP, TRANSFORM, CRS = src.read(1), src.transform, src.crs
    rows_, cols_ = pts['row'].to_numpy(), pts['col'].to_numpy()
    map_class = CLASS_MAP[rows_, cols_]
rule_reported = np.isin(map_class, [3, 4])
rule_every = np.isin(map_class, [3, 4, 5])

# The composites at the points' centre pixels, window by window.
xs, ys = rasterio.transform.xy(TRANSFORM, rows_, cols_)
point_refl = []
for d in WINDOWS:
    with rasterio.open(files[d]) as src:
        raw = np.array(list(src.sample(zip(xs, ys))), dtype='float32').T[band_order(src)]
        point_refl.append(to_reflectance(raw, src))
X, harvested = season_features(WINDOWS, lambda w: point_refl[w])
mismatch = (harvested != (map_class != 1)).sum()
print(f'{len(pts)} points: {sure.sum()} labelled ({truth[sure].sum()} burnt, {(~truth[sure]).sum()} not), '
      f'{(~sure).sum()} unsure or blank; {harvested.sum()} harvested.'
      + (f' WARNING: {mismatch} points differ from the class map on whether they were harvested.' if mismatch else ''))
train_pool = sure & harvested
print(f'The forest learns from the {train_pool.sum()} labelled, harvested points ({truth[train_pool].sum()} burnt).')

# The rule's scores here should match Notebook 02's validation table: a check that both read the same labels.
nb02_table = PROCESSED / f'validation_accuracy_{SUFFIX}.csv'
if nb02_table.exists():
    nb02_table = pd.read_csv(nb02_table)
    for name, pred in (('map as reported', rule_reported), ('every burn', rule_every)):
        row = nb02_table[nb02_table['rule'].str.contains(name, regex=False)
                         & (nb02_table['scored_on'] == 'all labelled points')]
        if len(row):
            ours, theirs = score(pred, sure)['f1'], float(row['f1'].iloc[0])
            print(f"Rule, {name}: F1 {ours:.3f} here and {theirs:.3f} in Notebook 02's validation table."
                  + ('' if abs(ours - theirs) < 1e-6 else ' WARNING: they differ; re-run Notebook 02 section 11.'))

table = pd.DataFrame(X, columns=FEATURES).assign(label=answer)
display(table[train_pool].groupby('label')[['peak_ndvi', 'min_nbr', 'max_mirbi', 'dark_nir', 'dark_swir2',
                                            'ndvi_after_dark']].median().round(3))

## 4. Train on half A, test on half B

The forest learns from the labelled, harvested points of half A and is scored on half B, which it never saw. This repeats the check of the rule in Notebook 02. Points never harvested count as not burnt for both methods. The forest gives every burn, early or late, so it is compared with both versions of the rule.

In [ ]:
model_a = forest().fit(ready(X[half_a & harvested]), truth[half_a & harvested])
rf_half = np.zeros(len(pts), bool)
rf_half[harvested] = model_a.predict(ready(X[harvested])).astype(bool)
rows = []
for name, pred in (('rule: map as reported', rule_reported), ('rule: every burn', rule_every),
                   ('forest trained on half A', rf_half)):
    rows.append({'method': name, 'scored_on': 'half B', **score(pred, half_b)})
test_b = pd.DataFrame(rows)
display(test_b.round(3))
for name, pred in (('map as reported', rule_reported), ('every burn', rule_every)):
    only_rule, only_rf, p = mcnemar(pred, rf_half, half_b)
    print(f'Half B, rule ({name}) against the forest: {only_rule} points only the rule gets right, {only_rf} only '
          f'the forest; exact McNemar p = {p:.2f}.')

## 5. Cross-validation on all labelled points

Half A alone is small, so the forest is also cross-validated: the labelled, harvested points are split into 5 folds, the forest is trained on 4 and predicts the 5th, and this is repeated 10 times with different splits. Every point is predicted only by forests that never saw it. The rule's thresholds were set before any label existed, so it is scored on all labelled points as it stands, as in Notebook 02.

In [ ]:
pool = np.where(train_pool)[0]
cv_runs, oof_prob, fold_models = [], np.zeros(len(pts)), []
splitter = RepeatedStratifiedKFold(n_splits=FOLDS, n_repeats=REPEATS, random_state=SEED)
started = time.time()
for k, (fit_i, test_i) in enumerate(splitter.split(pool, truth[pool])):
    model = forest(seed=SEED + k).fit(ready(X[pool[fit_i]]), truth[pool[fit_i]])
    run = k // FOLDS
    if len(cv_runs) <= run:
        cv_runs.append(np.zeros(len(pts), bool))
    prob = model.predict_proba(ready(X[pool[test_i]]))[:, 1]
    cv_runs[run][pool[test_i]] = prob >= 0.5
    oof_prob[pool[test_i]] += prob / REPEATS
    if k < FOLDS:
        fold_models.append((model, pool[test_i]))
print(f'{FOLDS * REPEATS} forests trained in {time.time() - started:.0f} s.')
cv_table = pd.DataFrame([score(pred, sure) for pred in cv_runs])
rows = [{'method': 'rule: map as reported', 'scored_on': 'all labelled points', **score(rule_reported, sure)},
        {'method': 'rule: every burn', 'scored_on': 'all labelled points', **score(rule_every, sure)}]
for stat in ('mean', 'min', 'max'):
    rows.append({'method': 'forest, cross-validated', 'scored_on': f'all labelled points, {stat} of {REPEATS} runs',
                 **getattr(cv_table, stat)().to_dict()})
cv_summary = pd.DataFrame(rows)
display(cv_summary.round(3))
pvalues = [mcnemar(rule_reported, run, sure)[2] for run in cv_runs]
print(f'McNemar against the map as reported, per run: p from {min(pvalues):.2f} to {max(pvalues):.2f}.')

## 6. Would more labels help?

The forest is trained on random subsets of growing size, using smaller forests of 200 trees to save time, and scored on held-out folds of about 30 points each. If F1 is still rising at the largest size, more labels would probably help; if it has flattened, they would not change the picture much. The rule is scored on the same folds, so the two lines compare like with like. F1 on 30 points is noisy, hence the error bars.

In [ ]:
rng = np.random.default_rng(SEED)
largest = int(len(pool) * (FOLDS - 1) / FOLDS)
sizes = [n for n in LEARNING_SIZES if n < largest] + [largest]
curve, rule_folds = [], []
for rep in range(LEARNING_REPEATS):
    for fit_i, test_i in StratifiedKFold(FOLDS, shuffle=True, random_state=SEED + rep).split(pool, truth[pool]):
        held, every_point = pool[test_i], np.ones(len(test_i), bool)
        rule_folds.append(scores(rule_reported[held], every_point, truth[held])[2])
        for n in sizes:
            chosen = rng.choice(pool[fit_i], size=min(n, len(fit_i)), replace=False)
            if truth[chosen].all() or not truth[chosen].any():
                continue
            pred = forest(seed=rep, trees=200).fit(ready(X[chosen]), truth[chosen]).predict(ready(X[held]))
            curve.append((n, scores(pred.astype(bool), every_point, truth[held])[2]))
learning = pd.DataFrame(curve, columns=['training_points', 'f1']).groupby('training_points')['f1'].agg(
    ['mean', 'std', 'count']).reset_index().rename(columns={'mean': 'f1_mean', 'std': 'f1_sd', 'count': 'runs'})
learning['rule_f1_mean_same_folds'] = np.mean(rule_folds)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.errorbar(learning['training_points'], learning['f1_mean'], yerr=learning['f1_sd'], marker='o', capsize=3,
            label='forest (mean ± 1 SD over folds and repeats)')
ax.axhline(np.mean(rule_folds), color='tab:red', linestyle='--', label='rule, map as reported (same folds)')
ax.set_xlabel('labelled points used for training')
ax.set_ylabel('F1 on held-out points')
ax.set_title('Learning curve')
ax.legend(fontsize=8, loc='lower right')
fig.savefig(FIGS / '01_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
display(learning.round(3))

## 7. What the forest relies on

Permutation importance (Breiman, 2001): on each held-out fold of the first cross-validation run, features are shuffled across the fold's points and the drop in F1 is measured, 10 times over. Many features carry the same information (the darkest view's NIR is also the lowest NIR after harvest), so a feature shuffled alone loses little while its partners stand in for it. The features are therefore also shuffled in five groups, each group's columns together:

| Group | Features |
|---|---|
| how dark the field turned after harvest | the lowest NBR, NIR, red and SWIR 2 and highest MIRBI after harvest, the NBR drop, and all bands and indices of the darkest view |
| the crop before harvest | green peak NDVI, NBR just before harvest |
| timing | harvest date, date of the darkest view, days between them |
| after the darkest view | NDVI and NIR change at the next view, highest NDVI after harvest, last NDVI of the season |
| number of clear views after harvest | how many clear views the field had from harvest on |

The scatter plot then shows the labelled points at their darkest view against the rule's NIR and NBR limits (its MIRBI limit is not drawn), and the cell prints how each of the rule's three limits does alone at that view.

In [ ]:
GROUPS = {
    'how dark the field turned after harvest': ['min_nbr', 'max_mirbi', 'min_nir', 'min_red', 'min_swir2', 'nbr_drop',
                                                'dark_red', 'dark_nir', 'dark_swir1', 'dark_swir2', 'dark_ndvi',
                                                'dark_nbr', 'dark_mirbi'],
    'the crop before harvest': ['peak_ndvi', 'pre_harvest_nbr'],
    'timing': ['harvest_day', 'dark_day', 'days_harvest_to_dark'],
    'after the darkest view': ['ndvi_after_dark', 'nir_rise_after_dark', 'max_ndvi_after_harvest', 'last_ndvi'],
    'number of clear views after harvest': ['views_after_harvest']}
assert sorted(sum(GROUPS.values(), [])) == sorted(FEATURES), 'every feature belongs to exactly one group'


def f1_drop(model, X_test, y_test, columns, rng, repeats=10):
    # Mean drop in F1 when `columns` are shuffled together across the test points.
    every_point = np.ones(len(y_test), bool)
    base = scores(model.predict(X_test).astype(bool), every_point, y_test)[2]
    drops = []
    for _ in range(repeats):
        shuffled = X_test.copy()
        shuffled[:, columns] = X_test[rng.permutation(len(X_test))][:, columns]
        drops.append(base - scores(model.predict(shuffled).astype(bool), every_point, y_test)[2])
    return np.mean(drops)


rng = np.random.default_rng(SEED)
sets = [('group', name, [FEATURES.index(f) for f in members]) for name, members in GROUPS.items()]
sets += [('feature', name, [k]) for k, name in enumerate(FEATURES)]
drops = np.zeros((len(fold_models), len(sets)))
for m, (model, test_i) in enumerate(fold_models):
    model.set_params(n_jobs=1)                      # faster for a few dozen points; the calls don't change
    for s, (_, _, columns) in enumerate(sets):
        drops[m, s] = f1_drop(model, ready(X[test_i]), truth[test_i], columns, rng)
importance = pd.DataFrame({'kind': [s[0] for s in sets], 'name': [s[1] for s in sets],
                           'features': [', '.join(FEATURES[k] for k in s[2]) for s in sets],
                           'f1_drop_mean': drops.mean(axis=0), 'f1_drop_sd': drops.std(axis=0)})
importance = importance.sort_values(['kind', 'f1_drop_mean'], ascending=[False, False], ignore_index=True)
display(importance[importance['kind'] == 'group'][['name', 'f1_drop_mean', 'f1_drop_sd']].round(3))
display(importance[importance['kind'] == 'feature'].head(8)[['name', 'f1_drop_mean', 'f1_drop_sd']].round(3))

groups = importance[importance['kind'] == 'group'].iloc[::-1]
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.barh(groups['name'], groups['f1_drop_mean'], xerr=groups['f1_drop_sd'], color='tab:gray', capsize=3)
ax.set_xlabel('drop in F1 when the group is shuffled (mean ± 1 SD over 5 folds)')
ax.set_title('What the forest relies on')
fig.savefig(FIGS / '02_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4.5))
x, y = X[:, FEATURES.index('dark_nir')], X[:, FEATURES.index('dark_nbr')]
ax.scatter(x[~sure & harvested], y[~sure & harvested], s=14, c='lightgray', label='unsure')
for value, colour, name in ((False, 'tab:blue', 'not burnt (no)'), (True, 'tab:red', 'burnt (yes)')):
    sel = train_pool & (truth == value)
    ax.scatter(x[sel], y[sel], s=18, c=colour, alpha=0.75, label=name)
ax.axvline(RULE['burn_nir_max'], color='k', linestyle='--', linewidth=1)
ax.axhline(RULE['burn_nbr_max'], color='k', linestyle='--', linewidth=1)
ax.set_xlabel('NIR at the darkest view after harvest')
ax.set_ylabel('NBR at the darkest view')
ax.set_title("Harvested points at their darkest view; dashed: the rule's NIR and NBR limits", fontsize=10)
ax.legend(fontsize=8)
fig.savefig(FIGS / '03_darkest_view.png', dpi=150, bbox_inches='tight')
plt.show()

# Each of the rule's three limits alone, applied to the darkest view after harvest.
dark = {name: X[:, FEATURES.index(f'dark_{name}')] for name in ('nir', 'nbr', 'mirbi')}
limits = {'NIR limit': dark['nir'] <= RULE['burn_nir_max'], 'NBR limit': dark['nbr'] <= RULE['burn_nbr_max'],
          'MIRBI limit': dark['mirbi'] >= RULE['burn_mirbi_min']}
limits['all three limits'] = limits['NIR limit'] & limits['NBR limit'] & limits['MIRBI limit']
print(f'At the darkest view after harvest, of the {train_pool.sum()} labelled, harvested points:')
for name, call in limits.items():
    print(f'  {name}: right on {(call == truth)[train_pool].sum()} ({(call & truth)[train_pool].sum()} of '
          f'{truth[train_pool].sum()} burnt caught, {(call & ~truth)[train_pool].sum()} unburnt called burnt)')

## 8. What the forest makes of the unsure points

From here on, one forest is trained on all labelled, harvested points. The `unsure` points were never used for training, so its calls on them are out of sample. They are not labels: the forest reads the same composites that left these points unsure, and it learnt only from clearer cases.

In [ ]:
model_all = forest().fit(ready(X[train_pool]), truth[train_pool])
unsure = ~sure & harvested
forest_prob = np.full(len(pts), np.nan)
forest_prob[train_pool] = oof_prob[train_pool]          # labelled: mean out-of-fold probability (section 5)
forest_prob[unsure] = model_all.predict_proba(ready(X[unsure]))[:, 1]
unsure_calls = pd.crosstab(np.where(rule_every[unsure], 'rule: burnt', 'rule: not burnt'),
                           np.where(forest_prob[unsure] >= 0.5, 'forest: burnt', 'forest: not burnt'))
display(unsure_calls)
print(f'{(~sure).sum()} unsure points, {unsure.sum()} of them harvested: the rule (every burn) calls '
      f'{rule_every[unsure].sum()} burnt and the forest {(forest_prob[unsure] >= 0.5).sum()}.')

## 9. The forest's map of the district

The forest from section 8 is run on every pixel of Notebook 02's map that was harvested, a block of rows at a time. Its map is then laid over the rule's, counting every burn on both sides:

| Value | Meaning |
|---|---|
| 1 | not a harvested crop |
| 2 | harvested, burnt in neither map |
| 3 | burnt in both maps |
| 4 | burnt in the rule's map only |
| 5 | burnt in the forest's map only |

The table compares the burnt areas with the estimate from the labels alone (Notebook 02), and measures where the disagreements lie: on the edge of a burnt patch in the rule's map (a pixel with at least one neighbour, diagonals included, that the rule doesn't call burnt), or in a patch under 1 ha. The figure shows the whole district and a close-up of the 5 km square with the most burns.

The validation points sit in the middle of uniform 3 × 3 blocks, so neither method's accuracy at field edges or in very small patches is measured. The comparison shows where the two disagree, not which is right there.

In [ ]:
comparison = pd.DataFrame()
if MAKE_MAP:
    started = time.time()
    height, width = CLASS_MAP.shape
    rule_burnt = np.isin(CLASS_MAP, [3, 4, 5])
    harvested_map = np.isin(CLASS_MAP, [2, 3, 4, 5])
    forest_burnt = np.zeros((height, width), bool)
    differ = 0                                            # pixels where this notebook and Notebook 02 disagree
    sources = [rasterio.open(files[d]) for d in WINDOWS]  # on whether they were harvested (should be 0)
    orders = [band_order(s) for s in sources]
    try:
        for r0 in range(0, height, CHUNK_ROWS):
            n_rows = min(CHUNK_ROWS, height - r0)
            block = Window(0, r0, width, n_rows)

            def read(w, block=block):
                raw = sources[w].read(window=block)[orders[w]].reshape(len(BANDS), -1)
                return to_reflectance(raw, sources[w])

            X_block, harvested_block = season_features(WINDOWS, read)
            in_map = harvested_map[r0:r0 + n_rows].reshape(-1)
            differ += int((harvested_block != in_map)[CLASS_MAP[r0:r0 + n_rows].reshape(-1) > 0].sum())
            todo = harvested_block & in_map
            if todo.any():
                calls = np.zeros(todo.size, bool)
                calls[todo] = model_all.predict(ready(X_block[todo])).astype(bool)
                forest_burnt[r0:r0 + n_rows] = calls.reshape(n_rows, width)
    finally:
        for s in sources:
            s.close()
    print(f'Forest run on {harvested_map.sum():,} harvested pixels in {time.time() - started:.0f} s.'
          + (f' WARNING: {differ:,} pixels differ from Notebook 02 on whether they were harvested.' if differ else ''))
    agreement = np.zeros(CLASS_MAP.shape, 'uint8')
    agreement[CLASS_MAP == 1] = 1
    agreement[harvested_map] = 2
    agreement[rule_burnt & forest_burnt] = 3
    agreement[rule_burnt & ~forest_burnt] = 4
    agreement[~rule_burnt & forest_burnt] = 5
    ha = abs(TRANSFORM.a * TRANSFORM.e) / 10_000

    # Where do they disagree? Edges of the rule's burnt patches (a pixel with a neighbour, diagonals included,
    # that the rule doesn't call burnt), and patches under 1 ha (pixels joined through any of their 8 neighbours).
    edge = rule_burnt & ~ndimage.binary_erosion(rule_burnt, structure=np.ones((3, 3)))
    patches, _ = ndimage.label(rule_burnt, structure=np.ones((3, 3)))
    small_patch = np.bincount(patches.ravel()) * ha < 1.0
    small_patch[0] = False                                # label 0 is the background, not a patch
    small = small_patch[patches]
    del patches
    both, rule_only = agreement == 3, agreement == 4

    def share(part, whole):
        return part.sum() / max(whole.sum(), 1)

    area_est = pd.read_csv(PROCESSED / f'validation_area_estimate_{SUFFIX}.csv').set_index('case')
    labels_row = area_est.loc['unsure left out']
    comparison = pd.DataFrame([
        {'measure': 'burnt in the rule map, every burn (ha)', 'value': rule_burnt.sum() * ha},
        {'measure': 'burnt in the rule map, as reported (ha)', 'value': np.isin(CLASS_MAP, [3, 4]).sum() * ha},
        {'measure': 'burnt in the forest map (ha)', 'value': forest_burnt.sum() * ha},
        {'measure': 'burnt in both (ha)', 'value': both.sum() * ha},
        {'measure': 'burnt in the rule map only (ha)', 'value': rule_only.sum() * ha},
        {'measure': 'burnt in the forest map only (ha)', 'value': (agreement == 5).sum() * ha},
        {'measure': 'burnt area from the labels alone (ha)', 'value': labels_row['burnt_ha_from_labels']},
        {'measure': 'burnt area from the labels, 95% CI low (ha)', 'value': labels_row['ci95_low_ha']},
        {'measure': 'burnt area from the labels, 95% CI high (ha)', 'value': labels_row['ci95_high_ha']},
        {'measure': 'share of rule-only pixels on a patch edge', 'value': share(rule_only & edge, rule_only)},
        {'measure': 'share of pixels burnt in both on a patch edge', 'value': share(both & edge, both)},
        {'measure': 'share of rule-only pixels in patches under 1 ha', 'value': share(rule_only & small, rule_only)},
        {'measure': 'share of pixels burnt in both in patches under 1 ha', 'value': share(both & small, both)},
        {'measure': "share of the rule's unconfirmed late burns the forest calls burnt",
         'value': share(forest_burnt & (CLASS_MAP == 5), CLASS_MAP == 5)}])
    display(comparison.round(3))

    profile = {'driver': 'GTiff', 'height': height, 'width': width, 'count': 1, 'dtype': 'uint8', 'crs': CRS,
               'transform': TRANSFORM, 'compress': 'deflate', 'nodata': 0}
    with rasterio.open(PROCESSED / f'ml_burn_agreement_{SUFFIX}.tif', 'w', **profile) as dst:
        dst.write(agreement, 1)
        dst.set_band_description(1, 'agreement: 1 not harvested, 2 neither, 3 both, 4 rule only, 5 forest only')

    # The whole district, and a close-up of the 5 km square with the most burnt pixels in either map.
    side = min(int(round(5000 / abs(TRANSFORM.a))), height, width)
    side_km = side * abs(TRANSFORM.a) / 1000
    rows_5k, cols_5k = height // side, width // side
    busy = (agreement[:rows_5k * side, :cols_5k * side] >= 3).reshape(rows_5k, side, cols_5k, side).sum(axis=(1, 3))
    r5, c5 = np.unravel_index(busy.argmax(), busy.shape)
    colours = ['#ffffff', '#e6e6e6', '#cfe3f2', '#2b2b2b', '#e6550d', '#3182bd']
    style = dict(cmap=ListedColormap(colours), vmin=0, vmax=5, interpolation='nearest')
    fig, (ax, ax_zoom) = plt.subplots(1, 2, figsize=(12, 7.5), gridspec_kw={'width_ratios': [width / height, 1]})
    ax.imshow(agreement, **style)
    ax.add_patch(Rectangle((c5 * side, r5 * side), side, side, fill=False, edgecolor='tab:red', linewidth=1.5))
    ax.set_title('Whole district')
    ax_zoom.imshow(agreement[r5 * side:(r5 + 1) * side, c5 * side:(c5 + 1) * side], **style)
    ax_zoom.set_title(f'Close-up of the red square: the {side_km:.0f} km square with the most burns')
    for a in (ax, ax_zoom):
        a.set_axis_off()
    fig.legend(handles=[Patch(color=c, label=t) for c, t in zip(colours[1:], (
        'not a harvested crop', 'harvested, burnt in neither', 'burnt in both', 'burnt in the rule map only',
        'burnt in the forest map only'))], loc='lower center', ncol=5, fontsize=9, frameon=False)
    fig.suptitle('Forest and rule compared, every burn on both sides, 2025 season (20 m pixels)')
    fig.tight_layout(rect=(0, 0.05, 1, 0.95))
    fig.savefig(FIGS / '04_agreement_map.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('MAKE_MAP is False: the district map is skipped.')

## 10. Outputs

The point table holds, for each of the 240 points, its label, both versions of the rule's call, the forest's probability of a burn (out of sample: the mean over cross-validation runs for labelled points, the forest of section 8 for `unsure` ones), the call of the forest trained on half A (for half B only) and the 23 features.

In [ ]:
in_half_b = (pts['half'] == 'B').to_numpy()
point_table = pts[['point_id', 'half', 'stratum']].assign(
    label=answer, harvested=harvested, rule_map_class=map_class, rule_map_as_reported=rule_reported,
    rule_every_burn=rule_every, forest_probability=forest_prob.round(3),
    forest_half_a_call=pd.Series(rf_half).where(in_half_b))
point_table = pd.concat([point_table, pd.DataFrame(X, columns=FEATURES).round(4)], axis=1)
point_table.to_csv(PROCESSED / f'ml_point_features_{SUFFIX}.csv', index=False)
pd.concat([test_b, cv_summary], ignore_index=True).to_csv(PROCESSED / f'ml_accuracy_{SUFFIX}.csv', index=False)
learning.to_csv(PROCESSED / f'ml_learning_curve_{SUFFIX}.csv', index=False)
importance.to_csv(PROCESSED / f'ml_feature_importance_{SUFFIX}.csv', index=False)
if len(comparison):
    comparison.to_csv(PROCESSED / f'ml_map_comparison_{SUFFIX}.csv', index=False)

mean = cv_summary.set_index(['method', 'scored_on'])
rf_mean = mean.loc[('forest, cross-validated', f'all labelled points, mean of {REPEATS} runs')]
rule_all = mean.loc[('rule: map as reported', 'all labelled points')]
print(f"On all labelled points: the rule (map as reported) F1 {rule_all['f1']:.3f}, overall accuracy "
      f"{rule_all['overall_accuracy']:.3f}; the cross-validated forest F1 {rf_mean['f1']:.3f}, overall accuracy "
      f"{rf_mean['overall_accuracy']:.3f}.")
print('Outputs written to', PROCESSED, 'and', FIGS)

### Reading the results
- **Same points, same scoring.** The rule's three thresholds were set before any label existed, so the every-burn rule's score on all labelled points is fair. The map as reported adds one choice made after looking at the labels, the 19 Nov cut-off, so its score may flatter it slightly. The forest's scores come only from points it was not trained on.
- **A tie is a result.** With fewer than 200 labels, differences of a few points of F1 are within chance: check the McNemar p-values and the spread across cross-validation runs before calling a winner.
- **What the map comparison can and can't say.** The labels show which method is more accurate at uniform blocks; the agreement map shows where the methods disagree, mostly edges and small patches, where neither has been checked.

### Caveats
- **Labels from imagery.** The forest learns from the same image-interpreted labels it is scored on, never from field visits.
- **One season, one district.** A forest trained on Sangrur 2025 may not transfer to another year or district without new labels; the rule's three thresholds are easier to check and carry over.
- **Late burns.** Only one labelled point lies among the rule's unconfirmed late burns (a `no`), so the forest's calls on those burns are not checked either.